# Metrics Demo

In [1]:
# Save the start time
import time
start_time = time.time()

# Allow crest to be imported
%cd -q ..

/Users/draghun1/miniconda3/envs/crest_cpu/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
from crest import DataServer

lstm_streamflow = DataServer.load('lstm_streamflow')

/Users/draghun1/miniconda3/envs/crest_cpu/lib/python3.10/site-packages/sklearn/utils/__init__.py:15: UserWarning: A NumPy version >=1.22.4 and <1.29.0 is required for this version of SciPy (detected version 1.22.3)
  from scipy.sparse import issparse


In [3]:
import matplotlib.pyplot as plt
import tensorflow as tf
import numpy as np

In [4]:
prcp_data = (lstm_streamflow['PRCP(mm/day)_maurer'].dropna(dim='date').to_numpy())

print(prcp_data)

feature_len = (prcp_data.shape)[0]
time_series_len = (prcp_data.shape)[1]

INP = 'precipitation'
OUT = 'streamflow'

print(feature_len, time_series_len)

[[ 0.          0.          0.         ... 39.56000137 10.57999992
   0.64999998]]
1 10501


In [5]:
def xy_rolling_window(data, window):
    ''' Generate a rolling window of input and target sets for the given data '''
    
    def rolling_window(data, size):
        window = lambda i: slice(i, i+size)
        result = [data[window(i)] for i in range(len(data) - size)]
        return np.stack(result, axis=0)
    
    x = rolling_window(data, window)
    y = data[window:]
    return x.reshape(len(x), window, -1), y

In [6]:
n_train = int(0.5*time_series_len)
n_test = n_train + int(0.3*time_series_len)
print(n_train, n_test)

train= prcp_data[0][:n_train]
test= prcp_data[0][n_train:n_test]
valid = prcp_data[0][n_test:]

n_seq = 60
xy_train = xy_rolling_window(train,n_seq)
xy_valid = xy_rolling_window(valid,n_seq)
xy_test = xy_rolling_window(test,n_seq)

5250 8400


In [7]:
from crest.data.loading.StructuredDataset import StructuredDataset

ds_train = StructuredDataset(*xy_train, labels=[INP, OUT])
ds_valid = StructuredDataset(*xy_valid, labels=[INP, OUT])
ds_test  = StructuredDataset(*xy_test,  labels=[INP, OUT])

In [9]:
from crest.data.batching.Batcher import Batcher

batch_kwargs = {
    'batch_size' : 200,            # 200 samples in each batch
    'features'   : [[INP], [OUT]], # [{'image': ...}, {'class': ...}]
    'repeat'     : True,           # Yield indefinitely
    'workers'    : 0               # Number of background processes
}

batch_train = Batcher(ds_train, **batch_kwargs)
batch_kwargs['shuffle'] = False    # Turn shuffle off for prediction
batch_valid = Batcher(ds_valid,  **batch_kwargs)
batch_test  = Batcher(ds_test ,  **batch_kwargs)

In [10]:
from tensorflow.keras.layers import LSTM, Dense, Input
from crest.model.HierarchalTensorGraph import HierarchalTensorGraph as HTG

def crest_layer(layer):
    """ Wrap a tensorflow layer in a HierarchalTensorGraph """
    graph = HTG(lambda d: {OUT: layer(d[INP])}, name=layer.name)
    #set inputs/outputs
    graph.inputs = {INP:None}
    graph.outputs = {OUT:None}
    return graph

layers = [crest_layer(layer) for layer in [
    LSTM(5, return_state=False, return_sequences=False)
]]

# Rename outputs to use as inputs (except the last layer)
for i in layers[:-1]:
    i.rename_io(outputs_map={OUT:INP})
    
# Create the CREST graph: ensure tensor shapes are specified for the inputs and outputs
graph = HTG(
    name='streamflow',
    inputs={INP : tf.TensorSpec(shape=[None, n_seq, feature_len])},
    outputs={OUT : tf.TensorSpec(shape=[None, n_seq, 5])}
)
print(list(map(graph.add_edge, ['input']+layers, layers+['output'])))

INFO:crest.model.HierarchalTensorGraph:Created HierarchalTensorGraph lstm
INFO:crest.model.HierarchalTensorGraph:Created HierarchalTensorGraph streamflow
INFO:crest.model.HierarchalTensorGraph:Representing HTG streamflow
INFO:crest.model.HierarchalTensorGraph:Representing HTG lstm
INFO:crest.model.HierarchalTensorGraph:Adding edge from input to HierarchalTensorGraph("lstm", id=13530855888) in HTG streamflow
INFO:crest.model.HierarchalTensorGraph:Representing HTG streamflow
INFO:crest.model.HierarchalTensorGraph:Getting node input in HTG streamflow
INFO:crest.model.HierarchalTensorGraph:Checking if node input is in HTG streamflow
INFO:crest.model.HierarchalTensorGraph:Created HierarchalTensorGraph input
INFO:crest.model.HierarchalTensorGraph:Representing HTG streamflow
INFO:crest.model.HierarchalTensorGraph:Representing HTG lstm
INFO:crest.model.HierarchalTensorGraph:Getting node HierarchalTensorGraph("lstm", id=13530855888) in HTG streamflow
INFO:crest.model.HierarchalTensorGraph:Getti

[None, None]


In [11]:
import tensorflow as tf

def reduce_mean(y_true, y_pred):
    return tf.reduce_mean(tf.abs(y_true - y_pred))

In [12]:
from crest.model.Model import Model

model = Model(graph)
model.metric.register('reduce_mean', reduce_mean)

INFO:crest.model.Model:Initializing CREST Model
INFO:crest.model.HierarchalTensorGraph:Calling HTG streamflow
INFO:crest.model.HierarchalTensorGraph:Checking if HTG streamflow is a basenode
INFO:crest.model.HierarchalTensorGraph:Getting all sources in HTG streamflow
INFO:crest.model.HierarchalTensorGraph:Getting all sinks in HTG streamflow
INFO:crest.model.HierarchalTensorGraph:Representing HTG streamflow
INFO:crest.model.HierarchalTensorGraph:Applying feature map to input in HTG streamflow
INFO:crest.model.HierarchalTensorGraph:Checking if HTG streamflow is a basenode
INFO:crest.model.HierarchalTensorGraph:Getting all nodes in HTG streamflow
INFO:crest.model.HierarchalTensorGraph:Getting node input in HTG streamflow
INFO:crest.model.HierarchalTensorGraph:Getting node lstm in HTG streamflow
INFO:crest.model.HierarchalTensorGraph:Getting node output in HTG streamflow
INFO:crest.model.HierarchalTensorGraph:Representing HTG streamflow
INFO:crest.model.HierarchalTensorGraph:Getting all sin

In [16]:
build_kwargs= {
    'optimizer' : 'Adam', 
    'loss'      : 'mse', 
    'metrics'   : ['mse']
}

model.compile(**build_kwargs)
model.fit(batch_train, **{
    'validation_data'  : batch_valid,
    'steps_per_epoch'  : 50, 
    'validation_steps' : 5,  
    'epochs'           : 5,
    'verbose'          : 1
})

INFO:crest.model.Model:Compiling Keras Model
INFO:crest.model.Model:Building Keras Model
INFO:crest.model.Model:Training Keras Model
INFO:crest.model.Model:Called _make_batcher, making batcher from dataset
INFO:crest.model.Model:Called _make_batcher, making batcher from dataset
INFO:crest.model.Model:Called _make_batcher, making batcher from dataset


Epoch 1/5
50/50 [==============================] - 1s 17ms/step - loss: 41.9916 - mse: 41.9916 - val_loss: 60.4107 - val_mse: 60.4107
Epoch 2/5
50/50 [==============================] - 1s 21ms/step - loss: 43.9802 - mse: 43.9802 - val_loss: 60.7809 - val_mse: 60.7809
Epoch 3/5
50/50 [==============================] - 1s 16ms/step - loss: 40.9807 - mse: 40.9807 - val_loss: 64.6464 - val_mse: 64.6464
Epoch 4/5
50/50 [==============================] - 1s 22ms/step - loss: 40.1883 - mse: 40.1883 - val_loss: 52.0437 - val_mse: 52.0437
Epoch 5/5
50/50 [==============================] - 1s 16ms/step - loss: 39.9603 - mse: 39.9603 - val_loss: 64.2621 - val_mse: 64.2621


In [17]:
saved_metrics = model.metric.to_json()

with open('saved_metrics.json', 'w') as f:
    f.write(saved_metrics)

In [18]:
with open('saved_metrics.json', 'r') as f:
    loaded_metrics = f.read()

# reset metrics
model.metric = model.metric.from_json(loaded_metrics)

In [19]:
batch_test  = Batcher(ds_test ,  **batch_kwargs)

with batch_test as batcher:
    res = model.evaluate(batch_test, verbose=1, return_dict=True)
    print(res)

INFO:crest.model.Model:Evaluating Keras Model
INFO:crest.model.Model:Called _make_batcher, making batcher from dataset


1/1 [==============================] - 0s 12ms/step - loss: 34.8129 - mse: 34.8129
{'loss': 34.812862396240234, 'mse': 34.812862396240234}
